# ***Tenno: Data Privacy Act***

## **Retrieval Preparation: FAISS Index Construction for Legal RAG System**

This notebook constructs the retrieval component of the Retrieval-Augmented Generation (RAG) pipeline. Previously extracted and chunked legal documents are loaded from the `Chunks` directory, embedded using a Sentence-Transformer model, and indexed using FAISS for efficient semantic similarity search.



The resulting FAISS index and associated metadata are saved to Google Drive for reuse across all model evaluation notebooks (Falcon-7B, Hermes-7B, and Qwen-7B). By isolating retrieval construction into a single notebook, we ensure consistent context retrieval across models while minimizing redundant computation and memory usage.

In [ ]:
!pip install -q faiss-cpu sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.2 MB/s eta 0:00:00


## **Mount Google Drive**

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


This cell mounts Google Drive to the Colab runtime, enabling access to the project directory and previously saved chunk files. All retrieval artifacts generated in this notebook will also be stored in Drive for reuse in subsequent model evaluation notebooks.

## **Define Project Paths**

---

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project"
CHUNKS_DIR = os.path.join(PROJECT_DIR, "Chunks")

chunks_file = os.path.join(CHUNKS_DIR, "all_documents_chunks.csv")
faiss_index_path = os.path.join(CHUNKS_DIR, "faiss_index.bin")
faiss_meta_path = os.path.join(CHUNKS_DIR, "faiss_metadata.json")

print("Chunks file:", chunks_file)

Chunks file: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Chunks/all_documents_chunks.csv


This cell defines the directory structure for the project and specifies the locations of the chunked dataset and the files where the FAISS index and metadata will be saved. Explicit path definitions improve reproducibility and ensure consistency across notebooks.

## **Load Chunked Documents**

---

In [ ]:
import pandas as pd

df = pd.read_csv(chunks_file)

print("Columns:", list(df.columns))
print("Total chunks:", len(df))

df.head(2)

Columns: ['chunk_id', 'text', 'source', 'page_number']
Total chunks: 198


,chunk_id,text,source,page_number
0,0,gov.ph http://www.gov.ph/2012/08/15/republic-a...,RA-10173-Data-Privacy-Act-of-2012.pdf,1
1,1,computer system or other similar device by or ...,RA-10173-Data-Privacy-Act-of-2012.pdf,2


The previously generated chunked dataset is loaded into a Pandas DataFrame. This dataset contains segmented legal text along with metadata such as source file and page number. Printing the column names allows verification of the text column that will be embedded in the next step.

## **Load Sentence Transformer Model**

---

In [ ]:
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
sentence_model = SentenceTransformer(model_name)

print(f"Sentence Transformer '{model_name}' successfully loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence Transformer 'all-MiniLM-L6-v2' successfully loaded.


The Sentence-Transformer model converts textual chunks into dense numerical embeddings. The selected model (all-MiniLM-L6-v2) balances computational efficiency with strong semantic representation, making it suitable for large-scale embedding tasks in a Colab environment.

## **Generate Embeddings**

---

In [ ]:
from tqdm import tqdm
import numpy as np

texts = df["text"].fillna("").tolist()

print("Generating embeddings...")

embeddings = sentence_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64
)

print("Embedding shape:", embeddings.shape)

Generating embeddings...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding shape: (198, 384)


Each chunk of legal text is encoded into a fixed-length embedding vector. These vectors capture semantic meaning and allow similarity comparison in vector space. Batch processing improves efficiency while preventing excessive memory usage.

## **Build FAISS Index**

---

In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS index built.")
print("Total vectors indexed:", index.ntotal)

FAISS index built.
Total vectors indexed: 198


A FAISS index is constructed using L2 distance for similarity search. The index stores all chunk embeddings, enabling efficient nearest-neighbor retrieval during question answering. Each vector corresponds to one legal text chunk.

## **Save FAISS Index and Metadata**

---

In [ ]:
import json

# Save FAISS index
faiss.write_index(index, faiss_index_path)

# Save metadata aligned with embeddings
metadata = df.to_dict(orient="records")

with open(faiss_meta_path, "w") as f:
    json.dump(metadata, f)

print("FAISS index saved to:", faiss_index_path)
print("Metadata saved to:", faiss_meta_path)

FAISS index saved to: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Chunks/faiss_index.bin
Metadata saved to: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Chunks/faiss_metadata.json


The FAISS index is saved as a binary file for later reuse. Additionally, chunk metadata is serialized into JSON format in the same order as the embeddings. This alignment ensures that retrieved vector indices can be accurately mapped back to their corresponding legal text chunks.

## **Retrieval Layer Successfully Constructed**

---



The FAISS index and associated metadata have been successfully generated and saved. This retrieval layer will be reused across all model evaluation notebooks to ensure consistent semantic search results and fair comparative analysis.